## Prompt Evaluation

This lesson looks at how prompts can be evaluated in a structured process.

In [2]:
# Import python's built-in regular expression library
import re
import os
from pathlib import Path
import anthropic
import json

def load_env_file(env_path=".env"):
    env_path = Path(env_path)

    if not env_path.exists():
        raise FileNotFoundError(f"Could not find .env file at: {env_path.resolve()}")

    with env_path.open("r") as file:
        for line in file:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            if "=" not in line:
                continue

            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")

load_env_file(".env")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
MODEL_NAME = os.environ.get("MODEL_NAME")

if not API_KEY:
    raise ValueError("Missing ANTHROPIC_API_KEY in .env")

if not MODEL_NAME:
    raise ValueError("Missing MODEL_NAME in .env")

client = anthropic.Anthropic(api_key=API_KEY)

print("Loaded API key and model name from .env")


Loaded API key and model name from .env


In [3]:
# creating helper fuctions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": MODEL_NAME,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [4]:
# function to generate test data set

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [5]:
# running the test data set function

dataset = generate_dataset()
print(dataset)

[{'task': 'Write a Python function that takes an AWS S3 bucket name and returns True if it follows AWS naming conventions (lowercase, 3-63 characters, no consecutive hyphens), False otherwise.'}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'."}, {'task': 'Write a regex pattern that matches valid AWS EC2 security group IDs (format: sg-followed by 8 or 17 hexadecimal characters).'}]


In [6]:
#saving the dataset

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)



## Moving on to building Prompt Testing Pipeline

In [7]:
# this is the dataset pipeline to evalute how good our prompts are
# the run_prompt function
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

# the run_test_case function

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

# run_eval function that coordinates the whole process

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [8]:
# running the evaluation by loading the dataset and running it through our functions
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)


In [9]:
# examing the results of the evaluation pipline.
    # each result contains 
        #output: The complete response from Claude
        #test_case: The original test case that was processed
        #score: The evaluation score (currently hardcoded)

print(json.dumps(results, indent=2))



[
  {
    "output": "# AWS S3 Bucket Name Validator\n\n```python\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates if a bucket name follows AWS S3 naming conventions.\n    \n    AWS S3 bucket naming rules:\n    - Must be between 3 and 63 characters long\n    - Must contain only lowercase letters, numbers, and hyphens\n    - Must start and end with a letter or number\n    - Cannot contain consecutive hyphens\n    - Cannot be formatted as an IP address (e.g., 192.168.1.1)\n    \n    Args:\n        bucket_name: The S3 bucket name to validate\n        \n    Returns:\n        True if the bucket name follows AWS conventions, False otherwise\n    \"\"\"\n    \n    # Check if bucket_name is a string\n    if not isinstance(bucket_name, str):\n        return False\n    \n    # Check length (3-63 characters)\n    if len(bucket_name) < 3 or len(bucket_name) > 63:\n        return False\n    \n    # Check if it contains only lowercase letters, numbers, a